# News Market Predictor — Exploratory Analysis
End-to-end walkthrough of data, features, and model results.

In [ ]:
import sys
sys.path.insert(0, '..')  # project root

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from sqlalchemy import text
from db.session import get_session
from db.models import Article, PriceRecord, SentimentScore, RAGFeature, Prediction
from config import TICKERS, SECTOR_MAP, setup_logging

setup_logging()
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
print('Setup complete.')

## 1 · Data Overview

In [ ]:
with get_session() as session:
    tables = {
        'articles':        session.query(Article).count(),
        'price_records':   session.query(PriceRecord).count(),
        'sentiment_scores': session.query(SentimentScore).count(),
        'rag_features':    session.query(RAGFeature).count(),
        'predictions':     session.query(Prediction).count(),
    }

    price_range = session.execute(
        text('SELECT MIN(date), MAX(date) FROM price_records')
    ).fetchone()

    article_range = session.execute(
        text('SELECT MIN(published_at), MAX(published_at) FROM articles')
    ).fetchone()

print('Row counts:')
for t, n in tables.items():
    print(f'  {t:<20s}: {n:>6,}')

print(f'\nPrice records : {price_range[0]} → {price_range[1]}')
print(f'Articles      : {article_range[0]} → {article_range[1]}')

In [ ]:
# Article count by ticker
with get_session() as session:
    rows = session.execute(
        text('SELECT ticker, COUNT(*) as n FROM articles GROUP BY ticker ORDER BY ticker')
    ).fetchall()

df_art = pd.DataFrame(rows, columns=['ticker', 'n'])
df_art['sector'] = df_art['ticker'].map(SECTOR_MAP)
sector_colors = {'Tech': '#4C72B0', 'Finance': '#55A868', 'Healthcare': '#C44E52',
                 'Energy': '#8172B2', 'Consumer': '#CCB974'}
colors = df_art['sector'].map(sector_colors)

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(df_art['ticker'], df_art['n'], color=colors, edgecolor='white', linewidth=0.5)
ax.set_title('Article count per ticker', fontsize=13)
ax.set_ylabel('Articles')
ax.tick_params(axis='x', rotation=45)

# Legend for sectors
from matplotlib.patches import Patch
handles = [Patch(facecolor=c, label=s) for s, c in sector_colors.items()]
ax.legend(handles=handles, loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

## 2 · Sentiment Distribution

In [ ]:
with get_session() as session:
    rows = session.execute(
        text('''
            SELECT a.ticker, s.score
            FROM sentiment_scores s
            JOIN articles a ON a.id = s.article_id
        ''')
    ).fetchall()

df_sent = pd.DataFrame(rows, columns=['ticker', 'score'])
df_sent['sector'] = df_sent['ticker'].map(SECTOR_MAP)
print(df_sent['score'].describe().round(4))

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(16, 3), sharey=True)
sectors = list(sector_colors.keys())

for ax, sector in zip(axes, sectors):
    data = df_sent[df_sent['sector'] == sector]['score']
    ax.hist(data, bins=20, color=sector_colors[sector], edgecolor='white', linewidth=0.4)
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title(sector, fontsize=10)
    ax.set_xlabel('Sentiment score')

axes[0].set_ylabel('Count')
fig.suptitle('FinBERT Sentiment Distribution by Sector', fontsize=13)
plt.tight_layout()
plt.show()

## 3 · RAG Feature Analysis

In [ ]:
with get_session() as session:
    rows = session.execute(
        text('''
            SELECT a.ticker, r.mean_analog_return, r.analog_hit_rate,
                   r.analog_volatility, r.n_analogs
            FROM rag_features r
            JOIN articles a ON a.id = r.article_id
            WHERE r.n_analogs > 0
        ''')
    ).fetchall()

df_rag = pd.DataFrame(rows,
    columns=['ticker', 'mean_analog_return', 'analog_hit_rate',
             'analog_volatility', 'n_analogs'])
df_rag['sector'] = df_rag['ticker'].map(SECTOR_MAP)
print(f'Articles with ≥1 analog: {len(df_rag)}')
print(df_rag[['mean_analog_return', 'analog_hit_rate', 'analog_volatility', 'n_analogs']]
      .describe().round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for sector in sectors:
    sub = df_rag[df_rag['sector'] == sector]
    ax.scatter(sub['mean_analog_return'], sub['analog_hit_rate'],
               c=sector_colors[sector], label=sector, alpha=0.6, s=40, edgecolors='none')

ax.axvline(0, color='gray', linewidth=0.7, linestyle='--')
ax.axhline(0.5, color='gray', linewidth=0.7, linestyle='--')
ax.set_xlabel('Mean Analog Return')
ax.set_ylabel('Analog Hit Rate')
ax.set_title('RAG Features: Mean Analog Return vs Hit Rate (colored by sector)')
ax.legend(fontsize=8)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=1))
plt.tight_layout()
plt.show()

## 4 · Feature Correlations

In [ ]:
# Rebuild the 13-dim feature matrix from the DB directly
from features.builder import build_dataset, FEATURE_NAMES
from rag.retriever import get_or_create_collection

collection = get_or_create_collection()

with get_session() as session:
    X, y, returns = build_dataset(session, collection)

df_feat = pd.DataFrame(X.numpy(), columns=FEATURE_NAMES)
df_feat['label'] = y.numpy()
print(f'Dataset shape: {df_feat.shape}')

In [ ]:
corr = df_feat.corr()['label'].drop('label').sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(6, 6))
full_corr = df_feat.corr()
im = ax.imshow(full_corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(full_corr.columns)))
ax.set_yticks(range(len(full_corr.columns)))
ax.set_xticklabels(full_corr.columns, rotation=45, ha='right', fontsize=7)
ax.set_yticklabels(full_corr.columns, fontsize=7)
plt.colorbar(im, ax=ax, fraction=0.046)
ax.set_title('Feature Correlation Matrix (incl. label)', fontsize=11)
plt.tight_layout()
plt.show()

print('\nTop correlations with label:')
print(corr.round(4).to_string())

## 5 · Model Results

In [ ]:
import torch
from sklearn.metrics import roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay
from model.architecture import NewsMarketClassifier, NewsMarketDataset
from model.train import get_dataloaders
from pathlib import Path

CHECKPOINT = Path('../model/checkpoints/best_model.pt')
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

model = NewsMarketClassifier.load(str(CHECKPOINT)).to(device)
_, _, test_loader = get_dataloaders(X, y, returns=returns)

model.eval()
all_probs, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        X_b = batch[0].to(device)
        probs = model(X_b).cpu().tolist()
        all_probs.extend(probs)
        all_labels.extend(batch[1].tolist())

preds = [1 if p >= 0.5 else 0 for p in all_probs]
print(f'Test samples: {len(all_labels)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Confusion matrix
cm = confusion_matrix(all_labels, preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['DOWN', 'UP'])
disp.plot(ax=axes[0], colorbar=False)
axes[0].set_title('Confusion Matrix — Test Set')

# ROC curve
fpr, tpr, _ = roc_curve(all_labels, all_probs)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='#4C72B0', lw=2, label=f'AUC = {roc_auc:.4f}')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()
print(f'ROC-AUC: {roc_auc:.4f}')